# 04 LoRA 怎么贴到 Qwen 上：代码里的 adapter 实现

LoRA 的核心直觉：不重训整本大模型，只在某些线性层旁边训练低秩小矩阵。代码里要找 `LoraConfig`、`target_modules`、`get_peft_model`。

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, textwrap, re

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts").exists():
    PROJECT_ROOT = Path(r"D:/coding/qwen lorar sft/qwen-local-lora-sft-dpo")

print("项目根目录:", PROJECT_ROOT)
print("学习原则: 本 notebook 默认只读项目文件；所有真实推理/训练开关默认 False。")

def read_text(rel):
    return (PROJECT_ROOT / rel).read_text(encoding="utf-8", errors="replace")

def show_file(rel, start=1, end=None):
    lines = read_text(rel).splitlines()
    if end is None:
        end = len(lines)
    print(f"--- {rel}:{start}-{end} ---")
    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:03d}: {lines[i-1]}")

def find_lines(rel, keyword, context=4):
    lines = read_text(rel).splitlines()
    hits = []
    for idx, line in enumerate(lines, start=1):
        if keyword in line:
            hits.append(idx)
    if not hits:
        print("没有找到:", keyword)
        return
    for hit in hits:
        start = max(1, hit - context)
        end = min(len(lines), hit + context)
        show_file(rel, start, end)
        print()

def read_jsonl(rel, n=3):
    rows = []
    with (PROJECT_ROOT / rel).open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
            if len(rows) >= n:
                break
    return rows

def count_jsonl(rel):
    with (PROJECT_ROOT / rel).open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

## 1. 训练脚本如何加载 base Qwen？

In [ ]:
find_lines("scripts/train_sft_lora.py", "AutoModelForCausalLM.from_pretrained", context=12)

这和推理脚本类似：先把完整 Qwen base model 加载出来。LoRA 不是替代 base model，而是在 base model 上加 adapter。

## 2. 从已有 adapter 继续训练，还是新建 LoRA？

In [ ]:
find_lines("scripts/train_sft_lora.py", "init_adapter_path", context=12)
find_lines("scripts/train_sft_lora.py", "PeftModel.from_pretrained", context=8)

这里对应项目里的重要经验：从头训练 v2 会回归旧 prompt；从已有好 adapter 低学习率续训 v3 更稳。所以脚本支持 `--init_adapter_path`。

## 3. 新建 LoRA adapter 的核心代码

In [ ]:
find_lines("scripts/train_sft_lora.py", "LoraConfig", context=16)
find_lines("scripts/train_sft_lora.py", "get_peft_model", context=8)

参数解释：`r` 是低秩矩阵的秩；`lora_alpha` 控制缩放；`lora_dropout` 减少过拟合；`target_modules` 指定 LoRA 贴到哪些层。本项目贴在注意力层和 MLP 层。

## 4. 为什么 LoRA 省显存？

因为 base model 大部分参数冻结，不更新梯度；训练时只更新 adapter 小参数。项目报告里 trainable ratio 大约 0.88%，也就是只训练不到 1% 的参数。

In [ ]:
find_lines("scripts/train_sft_lora.py", "print_trainable_parameters", context=5)

## 5. 你要能讲出的底层句子

> 本项目先加载 Qwen base model，然后用 PEFT 的 LoraConfig 指定 r、alpha、dropout 和目标线性层，再用 get_peft_model 把 LoRA adapter 挂到模型上。训练时主要更新 adapter 参数，因此适合 8GB 显存的本地实验。